# Quipu Extraction (NB 60)

One-shot scan + extraction pipeline. Builds `data/quipu_data.csv` (metadata table — one row per inscription) and `data/bodies/{root_txid}.bin` (raw bytes per inscription) so that all downstream analyses can be pure pandas with no RPC hammering.

Re-run this notebook to refresh after new inscriptions land on chain.

## Setup — RPC config + the 9 watched addresses

In [45]:
import os, sys, json, time
REPO = os.path.abspath('..')
sys.path.insert(0, REPO)
sys.path.insert(0, os.path.join(REPO, 'canonical'))

import pandas as pd
from colegio_tools import rpc_request

DATA_DIR    = os.path.join(REPO, 'data')
BODIES_DIR  = os.path.join(DATA_DIR, 'bodies')
os.makedirs(BODIES_DIR, exist_ok=True)

ADDRESSES = {
    '9xth7DcLGb1nACScMBeSfDCfghhLKF7yqs': 'bordado',
    'D6zKNnkupqRbkB9p5rwix8QiobQWJazjyX': 'apocrypha',
    'A7pfCe2Cw9JD2C4vEZbpDmUZJy7B2TaefV': 'ha',
    'AD28bxzxyrd3a4Qgad2VNQ2eN5Leg8ozuw': 'ca',
    'A3ShjwjsAE4ysM66EZJM3A28tPnL2jNDgC': 'multiman',
    'A3ABo52FjMJ57KSjbKyfe9aiKkH2jntXHY': 'test_multisig3',
    'DPy94XwsHvFpXfA2C6PjERknyNjQYacufZ': 'test1',
    'D6hcCyELYoMgPiMUfjGgBSHNSdZWrULupx': 'test2',
    'DPJAJuNW9ajjnEUy9RhYDjoMB9aFmkLdDb': 'test3',
}
ADDRESS_LIST = list(ADDRESSES.keys())
print(f'tip: {rpc_request("getblockcount")}')
for addr, label in ADDRESSES.items():
    print(f'  {label:16s} {addr}')

tip: 6215141
  bordado          9xth7DcLGb1nACScMBeSfDCfghhLKF7yqs
  apocrypha        D6zKNnkupqRbkB9p5rwix8QiobQWJazjyX
  ha               A7pfCe2Cw9JD2C4vEZbpDmUZJy7B2TaefV
  ca               AD28bxzxyrd3a4Qgad2VNQ2eN5Leg8ozuw
  multiman         A3ShjwjsAE4ysM66EZJM3A28tPnL2jNDgC
  test_multisig3   A3ABo52FjMJ57KSjbKyfe9aiKkH2jntXHY
  test1            DPy94XwsHvFpXfA2C6PjERknyNjQYacufZ
  test2            D6hcCyELYoMgPiMUfjGgBSHNSdZWrULupx
  test3            DPJAJuNW9ajjnEUy9RhYDjoMB9aFmkLdDb


## Step 1 — Scan: build `df_transactions` + `df_outputs`

Mirrors the quipu3.ipynb scan: pull all wallet txs once, then for each unique txid fetch the raw transaction (block height, vin, vout values, OP_RETURN) and populate the outputs frame with `spent_in` cross-references.

In [46]:
from colegio_tools import extract_op_return

# Pull every wallet event in one big request. listtransactions returns events,
# not unique txs — multi-address txs appear multiple times.
print('pulling wallet events...')
all_events = rpc_request('listtransactions', ['*', 200000, 0, True])
print(f'  {len(all_events)} events')

# Filter to txs that touch one of our watched addresses
txids_per_addr = {a: set() for a in ADDRESS_LIST}
for e in all_events:
    a = e.get('address')
    if a in txids_per_addr:
        txids_per_addr[a].add(e['txid'])

all_txids = set()
for s in txids_per_addr.values():
    all_txids |= s
print(f'  {len(all_txids)} unique txids across all watched addresses')

pulling wallet events...
  48354 events
  15300 unique txids across all watched addresses


In [47]:
# Fetch raw tx detail for each unique txid (expensive — runs once)
print(f'fetching {len(all_txids)} raw txs...')
detailed = []
blockheight_cache = {}
for i, txid in enumerate(all_txids):
    try:
        raw = rpc_request('getrawtransaction', [txid, 1])
    except Exception as e:
        print(f'  skip {txid[:12]}…: {e}')
        continue
    bh = raw.get('blockhash')
    if bh and bh in blockheight_cache:
        height = blockheight_cache[bh]
    elif bh:
        height = rpc_request('getblock', [bh])['height']
        blockheight_cache[bh] = height
    else:
        height = None
    op_ret = None
    # Find any OP_RETURN output (usually the last)
    for v in raw.get('vout', []):
        d = extract_op_return(v)
        if d:
            op_ret = d
            break
    detailed.append({
        'txid':         txid,
        'blockhash':    bh,
        'blockheight':  height,
        'blocktime':    raw.get('blocktime'),
        'inputs':       [f"{vin['txid']}:{vin['vout']}" for vin in raw.get('vin', []) if 'txid' in vin],
        'values':       [v['value'] for v in raw.get('vout', [])],
        'num_inputs':   len(raw.get('vin', [])),
        'num_outputs':  len(raw.get('vout', [])),
        'op_return':    op_ret,
    })
    if (i+1) % 5000 == 0:
        print(f'  {i+1} / {len(all_txids)}')

df_transactions = pd.DataFrame(detailed).sort_values(['blockheight','blocktime']).reset_index(drop=True)
print(f'df_transactions: {len(df_transactions)} rows')

fetching 15300 raw txs...
  5000 / 15300
  10000 / 15300
  15000 / 15300
df_transactions: 15300 rows


In [48]:
# Build df_outputs: one row per (txid, vout) with spent_in cross-reference.
# Convention: every output row of a tx carries the same op_return value
# (matches what identify_quipus expects from quipu3.ipynb).
rows = []
for _, tx in df_transactions.iterrows():
    for n in range(tx['num_outputs']):
        rows.append({
            'txout':       f"{tx['txid']}:{n}",
            'spent_in':    None,
            'value':       tx['values'][n] if n < len(tx['values']) else None,
            'op_return':   tx['op_return'],
            'blockheight': tx['blockheight'],
            'blocktime':   tx['blocktime'],
            'txid':        tx['txid'],
            'n':           n,
        })
df_outputs = pd.DataFrame(rows)

# Cross-link: for each tx's inputs, mark the consumed output's spent_in
txout_to_idx = {row['txout']: idx for idx, row in df_outputs.iterrows()}
for _, tx in df_transactions.iterrows():
    for inp in tx['inputs']:
        idx = txout_to_idx.get(inp)
        if idx is not None:
            df_outputs.at[idx, 'spent_in'] = tx['txid']

df_outputs = df_outputs.sort_values(['blockheight','blocktime']).reset_index(drop=True)
df_outputs['op_return'] = df_outputs['op_return'].fillna('')
df_outputs['spent_in']  = df_outputs['spent_in'].fillna('')
print(f'df_outputs: {len(df_outputs)} rows')

df_outputs: 31812 rows


## Step 2 — Identify quipu roots

A quipu root is a tx whose every output is spent by a subsequent tx carrying an OP_RETURN. Mirrors `identify_quipus` in `colegio_tools.py`.

In [49]:
from colegio_tools import identify_quipus, read_quipu

roots = identify_quipus(df_transactions, df_outputs)
print(f'{len(roots)} quipu root candidates')

66 quipu root candidates


## Step 3 — Walk each root and parse the header

For each root: `read_quipu(root, df_outputs)` returns `(header_hex, body_hex)`. We concat to get raw bytes, classify by type byte, parse per-type dimensions, and emit one row per inscription. Bodies go to `data/bodies/{root_txid}.bin`.

In [ ]:
TYPE_NAMES = {
    0x00: 'text', 0x03: 'image', 0x07: 'audio',
    0x0e: 'encrypted', 0x1d: 'identity',
    0xab: 'binding', 0xcc: 'cert',
    0xce: 'celestial', 0xee: 'estandarte',
}
TONE_NAMES = {0x00: 'ordinary', 0x01: 'affection', 0xff: 'reverence'}

def parse_dims(blob):
    """Return (dimensions_dict, title, header_length) for any v1 quipu.

    For images, body offset is back-computed from the declared (W, H, color,
    bit_depth) so historical title conventions (no pipes, |...|, | |...| |,
    cabeza padding) all parse consistently. The dimensions in the structural
    header are authoritative; everything between offset 12 and body_offset
    is the title region.
    """
    if len(blob) < 6 or blob[:4] != b'\xc1\xdd\x00\x01':
        return {}, '', 0
    t = blob[4]
    if t == 0x00:  # text
        hdr_end = 6
        title = ''
        if hdr_end < len(blob) and blob[hdr_end:hdr_end+1] == b'|':
            close = blob.find(b'|', hdr_end+1)
            if close > 0:
                title = blob[hdr_end+1:close].decode('utf-8', errors='replace')
                hdr_end = close + 1
        return {}, title, hdr_end
    if t == 0x03:  # image — back-compute body offset from declared dimensions
        if len(blob) < 12: return {}, '', 0
        color = blob[6]; W = (blob[7]<<8)|blob[8]; H = (blob[9]<<8)|blob[10]; bd = blob[11]
        dims = {'color': color, 'W': W, 'H': H, 'bit_depth': bd}
        if color not in (0x00, 0x01):
            return dims, '', 12  # non-canonical structural header
        ch = 1 if color == 0 else 3
        expected_body = (W * H * ch * bd + 7) // 8
        body_offset = len(blob) - expected_body
        if body_offset < 12:
            return dims, '', 12  # body math doesn't close
        # Lenient title extraction (canonical v1, May 2026):
        #   pipes  → first non-empty pipe-delimited field after whitespace strip
        #   else   → whole region UTF-8, truncate at first replacement char, strip
        text = blob[12:body_offset].decode('utf-8', errors='replace')
        if '|' in text:
            parts  = [p.strip() for p in text.split('|')]
            fields = [p for p in parts if p]
            title  = fields[0] if fields else ''
        else:
            cut = text.find('\ufffd')
            if cut >= 0:
                text = text[:cut]
            title = text.strip()
        return dims, title, body_offset
    if t == 0x0e:  # encrypted
        if len(blob) < 8: return {}, '', 0
        dims = {'sub_family': blob[6], 'variant': blob[7]}
        hdr_end = 8
        title = ''
        if hdr_end < len(blob) and blob[hdr_end:hdr_end+1] == b'|':
            close = blob.find(b'|', hdr_end+1)
            if close > 0:
                title = blob[hdr_end+1:close].decode('utf-8', errors='replace')
                hdr_end = close + 1
        return dims, title, hdr_end
    if t == 0xcc:  # cert
        if len(blob) < 8: return {}, '', 0
        sub = (blob[6]<<8) | blob[7]
        # Apply the same lenient pipe-title rule as text/image: first
        # pipe-delimited field of the body is the title.
        body = blob[8:]
        title = ''
        if body[:1] == b'|':
            close = body.find(b'|', 1)
            if close > 0:
                title = body[1:close].decode('utf-8', errors='replace').strip()
        return {'subtype': sub}, title, 8
    if t == 0xce:  # celestial v1
        if len(blob) < 12: return {}, '', 0
        kind = blob[6]; grouped = blob[7]; meta = blob[8]
        K = (blob[9]<<8) | blob[10]
        T = blob[11]
        title = blob[12:12+T].decode('utf-8', errors='replace') if T else ''
        return {'kind': kind, 'grouped': grouped, 'meta': meta, 'K': K}, title, 12 + T
    if t == 0xee:  # estandarte
        return {}, '', 6
    return {}, '', 0

def find_join_txid(root_txid, df_outputs):
    """Walk every strand to its terminus, find the common tx that spends them all."""
    termini_spenders = []
    n = 0
    while True:
        cur = f'{root_txid}:{n}'
        last_spender = None
        for _ in range(50):
            rows = df_outputs[df_outputs['txout'] == cur]
            if rows.empty: break
            sp = rows.iloc[0]['spent_in']
            if not sp:
                break
            last_spender = sp
            cur = f'{sp}:0'
        if last_spender is None:
            break
        termini_spenders.append(last_spender)
        n += 1
        if n > 32: break
    if not termini_spenders: return None
    from collections import Counter
    c = Counter(termini_spenders)
    most_common, count = c.most_common(1)[0]
    if count >= 2:
        return most_common
    return termini_spenders[-1]

In [51]:
# Build the lookup: which address received the root tx (output 0 typically)
# A quipu root sends all its outputs to a single address — the inscriber's address.
addr_per_root = {}
for root in roots:
    row = df_transactions[df_transactions['txid'] == root]
    if row.empty:
        addr_per_root[root] = None
        continue
    # Look up the address via the wallet's scriptPubKey on output 0
    try:
        raw = rpc_request('getrawtransaction', [root, 1])
        addr = raw['vout'][0].get('scriptPubKey', {}).get('addresses', [None])[0]
    except Exception:
        addr = None
    addr_per_root[root] = addr

print(f'looked up address for {sum(1 for v in addr_per_root.values() if v)} of {len(roots)} roots')

looked up address for 66 of 66 roots


In [52]:
# Walk each root, parse, write body file, accumulate rows.
# Skip identify_quipus heuristic false positives (no v1 magic) — they are
# regular multi-output sweeps that look quipu-shaped but aren't inscriptions.
rows = []
skipped_no_magic = []
for root in roots:
    try:
        hh, bh = read_quipu(root, df_outputs=df_outputs)
    except Exception as e:
        rows.append({'root_txid': root, 'notes': f'walk error: {e}'})
        continue
    blob = bytes.fromhex(hh + bh)
    if len(blob) < 6 or blob[:4] != b'\xc1\xdd\x00\x01':
        # Heuristic false positive — not a v1 quipu. Drop from quipu_data.csv.
        skipped_no_magic.append((root, blob[:16].hex() if blob else ''))
        continue

    t = blob[4]; tone = blob[5]
    dims, title, hdr_end = parse_dims(blob)
    type_name = TYPE_NAMES.get(t, f'unknown_0x{t:02x}')
    tone_name = TONE_NAMES.get(tone, f'unknown_0x{tone:02x}')

    addr = addr_per_root.get(root)
    label = ADDRESSES.get(addr, '(unknown)')

    tx_row = df_transactions[df_transactions['txid'] == root]
    blockheight = int(tx_row.iloc[0]['blockheight']) if not tx_row.empty and tx_row.iloc[0]['blockheight'] else None
    blocktime   = int(tx_row.iloc[0]['blocktime'])   if not tx_row.empty and tx_row.iloc[0]['blocktime']   else None

    join_txid = find_join_txid(root, df_outputs)

    # Body math check for images
    notes = ''
    if t == 0x03 and dims:
        ch = 1 if dims['color'] == 0 else 3
        expected_body = (dims['W'] * dims['H'] * ch * dims['bit_depth'] + 7) // 8
        actual_body = len(blob) - hdr_end
        if actual_body != expected_body:
            notes = f'image body mismatch: expect {expected_body} B, actual {actual_body} B'

    # Write body file
    body_path = os.path.join(BODIES_DIR, f'{root}.bin')
    with open(body_path, 'wb') as f:
        f.write(blob)

    rows.append({
        'root_txid':       root,
        'join_txid':       join_txid or '',
        'address':         addr or '',
        'label':           label,
        'type_byte':       f'0x{t:02x}',
        'type_name':       type_name,
        'tone':            f'0x{tone:02x}',
        'tone_name':       tone_name,
        'title':           title,
        'dimensions_json': json.dumps(dims, sort_keys=True),
        'total_bytes':     len(blob),
        'blockheight':     blockheight,
        'blocktime':       blocktime,
        'body_file':       f'bodies/{root}.bin',
        'notes':           notes,
    })

df_quipus = pd.DataFrame(rows).sort_values(['blockheight','root_txid']).reset_index(drop=True)
print(f'{len(df_quipus)} canonical inscriptions extracted')
print(f'{len(skipped_no_magic)} heuristic false positives skipped (no v1 magic)')
df_quipus.head(20)

33 canonical inscriptions extracted
33 heuristic false positives skipped (no v1 magic)


,root_txid,join_txid,address,label,type_byte,type_name,tone,tone_name,title,dimensions_json,total_bytes,blockheight,blocktime,body_file,notes
0,a2e9f2ebe1ceeae368b734fcd861549c813d5cfb009093...,4031dfd921fa5db207159d3cc171147e8916b7923f8e22...,D6zKNnkupqRbkB9p5rwix8QiobQWJazjyX,apocrypha,0x03,image,0xff,reverence,,"{""H"": 21360, ""W"": 8197, ""bit_depth"": 97, ""colo...",2611,4221666,1652377090,bodies/a2e9f2ebe1ceeae368b734fcd861549c813d5cf...,"image body mismatch: expect 6368823090 B, actu..."
1,c1542c10399a09a5471799133c472a6652d5cb3b7b421e...,4031dfd921fa5db207159d3cc171147e8916b7923f8e22...,D6zKNnkupqRbkB9p5rwix8QiobQWJazjyX,apocrypha,0x03,image,0xff,reverence,Sparkle🐈‍⬛MagicalCat🐈‍⬛✨💜Fovever💜✨,"{""H"": 64, ""W"": 64, ""bit_depth"": 5, ""color"": 0}",2630,4222137,1652407816,bodies/c1542c10399a09a5471799133c472a6652d5cb3...,
2,a01e8625d653f4a8686b5b9e20ca653e59ebcd8bf2ca2a...,4031dfd921fa5db207159d3cc171147e8916b7923f8e22...,D6zKNnkupqRbkB9p5rwix8QiobQWJazjyX,apocrypha,0x03,image,0xff,reverence,Peter Bea,"{""H"": 64, ""W"": 64, ""bit_depth"": 5, ""color"": 1}",7701,4224224,1652540740,bodies/a01e8625d653f4a8686b5b9e20ca653e59ebcd8...,
3,9e42c7ab6f47dadc0c36d22fb20f69c9db0daa5c719cd8...,4031dfd921fa5db207159d3cc171147e8916b7923f8e22...,D6zKNnkupqRbkB9p5rwix8QiobQWJazjyX,apocrypha,0x03,image,0xff,reverence,This was Peter on her blanket during a ride to...,"{""H"": 64, ""W"": 128, ""bit_depth"": 5, ""color"": 1}",15472,4240897,1653607139,bodies/9e42c7ab6f47dadc0c36d22fb20f69c9db0daa5...,
4,dcd31fa39202c9018dfb28e17da108dd3bd6b98ffc1cd3...,4031dfd921fa5db207159d3cc171147e8916b7923f8e22...,D6zKNnkupqRbkB9p5rwix8QiobQWJazjyX,apocrypha,0x03,image,0xff,reverence,Sun Face,"{""H"": 144, ""W"": 144, ""bit_depth"": 5, ""color"": 1}",38902,4241097,1653620310,bodies/dcd31fa39202c9018dfb28e17da108dd3bd6b98...,
5,014123b21a99b50e28219522af50a7a970dd3f8feeb0dd...,4031dfd921fa5db207159d3cc171147e8916b7923f8e22...,D6zKNnkupqRbkB9p5rwix8QiobQWJazjyX,apocrypha,0x03,image,0xff,reverence,Paco was a kitten I found on the side of the r...,"{""H"": 120, ""W"": 168, ""bit_depth"": 4, ""color"": 1}",30430,4244547,1653840768,bodies/014123b21a99b50e28219522af50a7a970dd3f8...,
6,d68175766b70f7163aec93e5a4e81480a6c6dd51d05773...,4031dfd921fa5db207159d3cc171147e8916b7923f8e22...,D6zKNnkupqRbkB9p5rwix8QiobQWJazjyX,apocrypha,0x0e,encrypted,0x03,unknown_0x03,,"{""sub_family"": 0, ""variant"": 0}",2800,4251800,1654304850,bodies/d68175766b70f7163aec93e5a4e81480a6c6dd5...,
7,d0209a0f85872d6826c58bc23fab37c8b21feb22c15a5a...,4031dfd921fa5db207159d3cc171147e8916b7923f8e22...,D6zKNnkupqRbkB9p5rwix8QiobQWJazjyX,apocrypha,0x0e,encrypted,0x03,unknown_0x03,,"{""sub_family"": 1, ""variant"": 0}",15655,4252445,1654346431,bodies/d0209a0f85872d6826c58bc23fab37c8b21feb2...,
8,89b51b4852b0e80f49cdb229d85ef4757d943c9fe4ba62...,4031dfd921fa5db207159d3cc171147e8916b7923f8e22...,D6zKNnkupqRbkB9p5rwix8QiobQWJazjyX,apocrypha,0x0e,encrypted,0x0e,unknown_0x0e,,"{""sub_family"": 13, ""variant"": 124}",112,4267974,1655342056,bodies/89b51b4852b0e80f49cdb229d85ef4757d943c9...,
9,f278e466012fb78422834742c6440c935f4cc2ef64e722...,4031dfd921fa5db207159d3cc171147e8916b7923f8e22...,D6zKNnkupqRbkB9p5rwix8QiobQWJazjyX,apocrypha,0x0e,encrypted,0x0e,unknown_0x0e,,"{""sub_family"": 13, ""variant"": 124}",112,4270461,1655501160,bodies/f278e466012fb78422834742c6440c935f4cc2e...,


## Step 3b — Tag canonical vs pre-canonical compliance

Run strict header-only checks per type and tag each inscription with `canonical_status`:
- `canonical_v1` — bytes match v1 spec exactly
- `pre_canonical` — known historical divergence (see pre_canonical_inscriptions memory)
- `not_yet_canonicalized` — type has no canonical reader yet (e.g. 0x1d identity, 0x0c precursor)

In [53]:
STRICT_CHECKS = {
    'text':      lambda b: (b[5] in (0x00,0x01,0xff) and (len(b)<=6 or b[6:7] != b'|' or b.find(b'|',7) >= 0)),
    'cert':      lambda b: (len(b)>=8 and b[5] in (0x00,0xff) and ((b[6]<<8)|b[7]) in (0x0001,0x0002)),
    'encrypted': lambda b: (len(b)>=8 and b[5] in (0x00,0x01,0xff) and b[6] in (0xae,0xec,0x0d)),
    'celestial': lambda b: (len(b)>=12 and b[5] in (0x00,0x01,0xff) and b[6] in (0x00,0x01) and b[7] in (0x00,0x01) and b[8] in (0x00,0x01)),
}

def check_image_strict(blob):
    # Canonical v1 with the lenient title rule: structural header must be
    # valid and body math must close. Title region (between offset 12 and
    # body start) is any of the v1-canonical forms (pipe, no-pipe, padded).
    if len(blob) < 12: return False
    if blob[5] not in (0x00, 0x01, 0xff): return False  # tone
    if blob[6] not in (0x00, 0x01):       return False  # color
    bd = blob[11]
    if not (1 <= bd <= 8):                return False
    W = (blob[7]<<8)|blob[8]; H = (blob[9]<<8)|blob[10]
    if W == 0 or H == 0:                  return False
    ch = 1 if blob[6] == 0 else 3
    expected = (W * H * ch * bd + 7) // 8
    body_offset = len(blob) - expected
    return body_offset >= 12

STRICT_CHECKS['image'] = check_image_strict

statuses = []
for _, row in df_quipus.iterrows():
    tname = row['type_name']
    if tname.startswith('unknown_'):
        statuses.append('not_yet_canonicalized')
        continue
    if tname == 'identity':
        # 0x1d identity type doesn't have a canonical reader yet
        statuses.append('not_yet_canonicalized')
        continue
    if tname not in STRICT_CHECKS:
        statuses.append('not_yet_canonicalized')
        continue
    bpath = f'data/{row["body_file"]}'
    if not os.path.exists(bpath):
        statuses.append('pre_canonical')  # we couldn't write the body — non-canonical
        continue
    blob = open(bpath, 'rb').read()
    statuses.append('canonical_v1' if STRICT_CHECKS[tname](blob) else 'pre_canonical')

df_quipus['canonical_status'] = statuses

# Print summary
from collections import Counter
counts = Counter(statuses)
print(f'Canonical compliance:')
for k, v in counts.most_common():
    print(f'  {k:25s} {v}')
print()
print('Pre-canonical inscriptions:')
for _, r in df_quipus[df_quipus['canonical_status'] == 'pre_canonical'].iterrows():
    print(f'  {r["root_txid"][:12]}…  {r["type_name"]:9s}  {r["label"]:14s}  {r["title"] if isinstance(r["title"], str) else "(no title)"}')

Canonical compliance:
  pre_canonical             31
  not_yet_canonicalized     2

Pre-canonical inscriptions:
  a2e9f2ebe1ce…  image      apocrypha       
  c1542c10399a…  image      apocrypha       Sparkle🐈‍⬛MagicalCat🐈‍⬛✨💜Fovever💜✨
  a01e8625d653…  image      apocrypha       Peter Bea
  9e42c7ab6f47…  image      apocrypha       This was Peter on her blanket during a ride to the country
  dcd31fa39202…  image      apocrypha       Sun Face
  014123b21a99…  image      apocrypha       Paco was a kitten I found on the side of the road in a drain ditch.
 The first night after taking him home he slept in a bag. 
 He was named after fashion designer Paco Rabanne
  d68175766b70…  encrypted  apocrypha       
  d0209a0f8587…  encrypted  apocrypha       
  89b51b4852b0…  encrypted  apocrypha       
  f278e466012f…  encrypted  apocrypha       
  aa0c3ea6b38b…  image      apocrypha       Dr. Doeg en Buenos Aires
  1ec0ee9b27d6…  cert       bordado         
  62e8f7355468…  text       ca         

In [54]:
# Save the metadata
csv_path = os.path.join(DATA_DIR, 'quipu_data.csv')
df_quipus.to_csv(csv_path, index=False)
print(f'wrote {csv_path} ({os.path.getsize(csv_path)} bytes)')

# Body files summary
body_files = [f for f in os.listdir(BODIES_DIR) if f.endswith('.bin')]
total_body_bytes = sum(os.path.getsize(os.path.join(BODIES_DIR, f)) for f in body_files)
print(f'{len(body_files)} body files, total {total_body_bytes/1024:.1f} KB')

# Also save a SLIM tx-inputs table (just txid + JSON-encoded inputs list)
# so the funding-edges cell can run on a fresh kernel without re-scanning
# or relying on ~/Desktop CSVs.
import json as _json
slim = df_transactions[['txid', 'inputs']].copy()
slim['inputs'] = slim['inputs'].apply(_json.dumps)
slim_path = os.path.join(DATA_DIR, 'tx_inputs.csv')
slim.to_csv(slim_path, index=False)
print(f'wrote {slim_path} ({os.path.getsize(slim_path)} bytes, {len(slim)} txs)')

wrote /Users/anthonyschultz/Desktop/Colegio_Invisible/data/quipu_data.csv (12831 bytes)
33 body files, total 1180.5 KB
wrote /Users/anthonyschultz/Desktop/Colegio_Invisible/data/tx_inputs.csv (2215975 bytes, 15300 txs)


## Step 4 — Summary by type and address

In [55]:
# Group by type + label
summary = df_quipus.groupby(['label', 'type_name']).size().unstack(fill_value=0)
print('--- inscriptions per (label, type) ---')
print(summary)

print('\n--- divergences flagged ---')
div = df_quipus[df_quipus['notes'].str.len() > 0]
if div.empty:
    print('  none — every inscription matches v1 spec')
else:
    for _, r in div.iterrows():
        print(f'  {r["root_txid"][:12]}…  type={r["type_byte"]}  {r["notes"]}')

--- inscriptions per (label, type) ---
type_name  celestial  cert  encrypted  identity  image  text  unknown_0x0c
label                                                                     
apocrypha          2     0          8         1      8     2             0
bordado            0     2          0         0      3     0             1
ca                 0     0          0         0      1     1             0
ha                 0     0          0         0      1     1             0
multiman           0     0          1         0      0     1             0

--- divergences flagged ---
  a2e9f2ebe1ce…  type=0x03  image body mismatch: expect 6368823090 B, actual 2599 B


## Query examples (no RPC after this point)

Re-load `quipu_data.csv` from any other notebook and query freely:

```python
import pandas as pd
df = pd.read_csv('data/quipu_data.csv')

# All celestial inscriptions
df[df['type_name'] == 'celestial']

# All bordado certificates
df[(df['label']=='bordado') & (df['type_name']=='cert')]

# Read a specific inscription's bytes
row = df[df['title'] == 'Sky of al-Jawza'].iloc[0]
blob = open(f'data/{row["body_file"]}', 'rb').read()
```

## Step 5 — Compute funding edges (`data/quipu_edges.csv`)

Self-contained cell — loads `data/quipu_data.csv` from disk, fetches each root tx via RPC to read its inputs, identifies which inputs come from other quipus' join transactions, and writes `data/quipu_edges.csv` with one row per (source_quipu, consumer_quipu) funding edge.

Designed to be runnable on its own after re-opening the notebook (does not depend on `df_transactions` in kernel). The 33 root-tx RPC calls take a few seconds total.

In [57]:
# Self-contained: read quipu_data.csv + tx_inputs.csv from disk, then for
# each quipu's root walk BACKWARD through its input ancestry until hitting
# another quipu's root or join. Adds keydrop -> target dotted edges too.
# Designed to run on a fresh kernel with no other cells run first.

import os, sys, pandas as pd, json
REPO = os.path.abspath('..')
sys.path.insert(0, REPO)
sys.path.insert(0, os.path.join(REPO, 'canonical'))

DATA_DIR = os.path.join(REPO, 'data')

df_quipus_local = pd.read_csv(os.path.join(DATA_DIR, 'quipu_data.csv'))
df_quipus_local = df_quipus_local[df_quipus_local['root_txid'].notna()].copy()
all_roots    = set(df_quipus_local['root_txid'])
joins        = set(df_quipus_local['join_txid'].dropna())
join_to_root = {r['join_txid']: r['root_txid'] for _, r in df_quipus_local.iterrows()
                if isinstance(r.get('join_txid'), str) and r['join_txid']}
quipu_struct = all_roots | joins

# Load tx_inputs from disk (saved by NB 60's main extraction step)
tx_inputs_path = os.path.join(DATA_DIR, 'tx_inputs.csv')
tx_inputs = {}
if os.path.exists(tx_inputs_path):
    print(f'loading tx_inputs from {tx_inputs_path}')
    df_ti = pd.read_csv(tx_inputs_path)
    for _, r in df_ti.iterrows():
        try:
            raw_inputs = json.loads(r['inputs'])
        except Exception:
            raw_inputs = []
        parsed = [tuple(s.rsplit(':', 1)) for s in raw_inputs]
        tx_inputs[r['txid']] = [(t, int(v)) for t, v in parsed if t]
else:
    raise FileNotFoundError(
        f'{tx_inputs_path} not found — run NB 60 cells 4–15 first to generate it.'
    )

def trace_back(root_txid, max_hops=15):
    """BFS backward through input ancestry, stopping at any tx not in
    our wallet's tx_inputs (those branches lead outside the quipu graph)."""
    seen = set()
    queue = [(root_txid, 0)]
    hits = []
    while queue:
        txid, hops = queue.pop(0)
        if hops > max_hops or txid in seen:
            continue
        seen.add(txid)
        if hops > 0 and txid in quipu_struct:
            hits.append((txid, hops))
            continue
        ancestors = tx_inputs.get(txid)
        if ancestors is None:
            # Tx not in wallet scan → outside the quipu graph; stop this branch.
            continue
        for prev_txid, _ in ancestors:
            if prev_txid not in seen:
                queue.append((prev_txid, hops + 1))
    return hits

# === Funding edges (backward walk through input ancestry) ===
funding = []
for _, q in df_quipus_local.iterrows():
    consumer = q['root_txid']
    for anc, hops in trace_back(consumer):
        src_root = anc if anc in all_roots else join_to_root.get(anc)
        if src_root and src_root != consumer:
            funding.append({
                'source_quipu':   src_root,
                'consumer_quipu': consumer,
                'hops':           hops,
                'kind':           'funding',
            })

# === Keydrop -> target dotted edges (encrypted sub_family 0x0d) ===
from encrypted import read_encrypted_quipu
keydrop_edges = []
for _, q in df_quipus_local.iterrows():
    if q['type_name'] != 'encrypted':
        continue
    dims = json.loads(q['dimensions_json'] or '{}')
    if dims.get('sub_family') != 0x0d:
        continue
    blob_path = os.path.join(DATA_DIR, q['body_file'])
    if not os.path.exists(blob_path):
        continue
    blob = open(blob_path, 'rb').read()
    try:
        parsed = read_encrypted_quipu(blob[:8], blob[8:])
    except Exception as e:
        print(f'  keydrop parse failed for {q["root_txid"][:8]}: {e}')
        continue
    for d in parsed.get('drops', []):
        ref = d.get('ref_txid')
        if ref in all_roots:
            keydrop_edges.append({
                'source_quipu':   q['root_txid'],
                'consumer_quipu': ref,
                'hops':           0,
                'kind':           'keydrop',
            })

all_edges = funding + keydrop_edges
df_edges = pd.DataFrame(all_edges)
df_edges.to_csv(os.path.join(DATA_DIR, 'quipu_edges.csv'), index=False)

print(f'\n{len(funding)} funding edges + {len(keydrop_edges)} keydrop edges')

print('\n--- funding (backward through consolidation chains) ---')
for e in funding:
    s = df_quipus_local[df_quipus_local['root_txid'] == e['source_quipu']].iloc[0]
    d = df_quipus_local[df_quipus_local['root_txid'] == e['consumer_quipu']].iloc[0]
    st = s['title'] if isinstance(s['title'], str) else '(no title)'
    dt = d['title'] if isinstance(d['title'], str) else '(no title)'
    print(f'  {e["source_quipu"][:8]}... "{st[:25]}" --{int(e["hops"])}hop--> {e["consumer_quipu"][:8]}... "{dt[:25]}"')

print('\n--- keydrop (unlocks) ---')
for e in keydrop_edges:
    s = df_quipus_local[df_quipus_local['root_txid'] == e['source_quipu']].iloc[0]
    d = df_quipus_local[df_quipus_local['root_txid'] == e['consumer_quipu']].iloc[0]
    dt = d['title'] if isinstance(d['title'], str) else '(no title)'
    print(f'  {e["source_quipu"][:8]}... (keydrop) ==unlocks==> {e["consumer_quipu"][:8]}... "{dt[:25]}"')

loading tx_inputs from /Users/anthonyschultz/Desktop/Colegio_Invisible/data/tx_inputs.csv
  keydrop parse failed for 89b51b48: keydrop body truncated reading drop 0 name
  keydrop parse failed for f278e466: keydrop body truncated reading drop 0 name

27 funding edges + 2 keydrop edges

--- funding (backward through consolidation chains) ---
  a2e9f2eb... "(no title)" --2hop--> c1542c10... "Sparkle🐈‍⬛MagicalCat🐈‍⬛✨💜"
  c1542c10... "Sparkle🐈‍⬛MagicalCat🐈‍⬛✨💜" --2hop--> a01e8625... "Peter Bea"
  a01e8625... "Peter Bea" --4hop--> 9e42c7ab... "This was Peter on her bla"
  9e42c7ab... "This was Peter on her bla" --3hop--> dcd31fa3... "Sun Face"
  dcd31fa3... "Sun Face" --2hop--> 014123b2... "Paco was a kitten I found"
  014123b2... "Paco was a kitten I found" --4hop--> d6817576... "(no title)"
  d6817576... "(no title)" --3hop--> d0209a0f... "(no title)"
  d0209a0f... "(no title)" --3hop--> 89b51b48... "(no title)"
  89b51b48... "(no title)" --2hop--> f278e466... "(no title)"
  f278e466... "